In [0]:
CREATE WIDGET TEXT end_date DEFAULT '2025-12-31';

In [0]:
select distinct PATIENT_ID from com_raw.kom_medical_events where PROCEDURE_CODE = 'J1743' or NDC11 in ('54092070001','540920700')
union 
select distinct patient_id from com_edp_prd.com_raw.kom_pharmacy_events where ndc11 in ('54092070001','540920700')

In [0]:
-- CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary AS
WITH
-- -----------------------------
-- Eligibility (Specified + Incremental Unspecified)
-- -----------------------------
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-12-31'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-12-31'
),
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-12-31'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-12-31'
),
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
-- elaprase_patients as (select distinct PATIENT_ID from com_raw.kom_medical_events where PROCEDURE_CODE in ('J1743') or NDC11 in ('54092070001','540920700')
-- union 
-- select distinct patient_id from com_edp_prd.com_raw.kom_pharmacy_events where ndc11 in ('54092070001','540920700')),
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '2025-12-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
                                --  WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366', 'S9357','S9379',
                                --  '38206','38230','38232','38240','38241','38242','38243','38250') and PATIENT_ID not in (select distinct PATIENT_ID from elaprase_patients)
                                --  WHERE PROCEDURE_CODE IN ('J1743')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'
    ) t
),
MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '2025-12-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'
    ) t
),
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
)
select *
from eligible_patients

In [0]:
with t1 as (
  select distinct patient_id, (year(current_date)-year(patient_yob)) as age
  from (SELECT DISTINCT patient_id, patient_yob
    FROM (
      SELECT
        PATIENT_ID AS patient_id,
        PATIENT_YOB AS patient_yob,
        row_number() OVER (PARTITION BY PATIENT_ID ORDER BY PATIENT_YOB DESC) AS rn
      FROM com_raw.kom_patient_demographics
    )
    WHERE rn = 1)
)

select *
from t1
where patient_id in ('00P0S2C0', '1181555', '020CLMY5', '03DRQPXS', '06NQHE5W', '06WGTFBV', '077VYZSG', '07F040KL', '0CFBS3QP', '0FD2MVJH', '0JXVTG3V', '0KPZPXKH', '0QMXBD93', '0RQ8WER6', '0RT0BC16', '0VQR3EKL', '15RV550S', '165FCRS8', '1ETSLF32', '1G84ST3K', '1HNFJ12Q', '1K7D3CZ4', '1PGKGH5D', '1TKZ01BN', '1WTYQP0W', '1X7KPMBS', '2193XXGH', '28KXJ10D', '2B8E9JSR', '2H51L6TJ', '2J7TTMMJ', '2L9ZH96X', '2TQZ1H93', '2W1LYFK2', '2XPN1XBS', '33PEP1TE', '357ZX8QE', '35RJEZJM', '36LNY5TR', '39BDW50Z', '3EN349ET', '3KFLJ9H5', '3L66Z8WX', '3NSV0MXJ', '42D8R98N', '47DCKLNL', '47NW3JF6', '47TME7CG', '4J55LN3N', '4K210EEE', '4NNDX85F', '4Q9YWTW2', '4R9JZEPZ', '4RC079JT', '4XK3PWRM', '4XQDPX1J', '508GEBRX', '52028LXY', '52ZYHDNZ', '55MXFFR4', '56Q7B70X', '58D46PD2', '596XQZSB', '5B1S5Y6M', '5B5YJ0BJ', '5B9F3PEP', '5D4NL3WG', '5GXK95K8', '5HREMYP4', '5KPZNFGX', '5S98ZESL', '5SR3QR8X', '5VKHE13W', '5WQ44ZEV', '5YQL634J', '6513VL4S', '6B0EWF5Q', '6KB07SBV', '6LRJE8F5', '6MGD17QW', '6MJEMWFT', '6Q831L53', '6XZ63TZJ', '6YWE9Z2W', '72R9ZN63', '73YZJ6P0', '755R9342', '78VMNREV', '7BDN8F8G', '7F3HFLSR', '7FZHNZ49', '7H3SQSYK', '7J3SLYFW', '7K2MLVSZ', '7M0D8HCK', '7MKVQFF5', '7N8GKT2J', '7SBZKMP0', '7TH16LJF', '7W3FYDPQ', '80YQPQ2T', '84HGEGMM', '85NWTEG1', '87CS8ZN7', '8B1MFBXC', '8KDEH0H8', '8LSZ647B', '8ML7WJG8', '8NNK4CH8', '8QR2GY9W', '8TQK891K', '8W4Z8SHZ', '8WW54LY8', '8ZHEJWP3', '905ESMNC', '92M2CVGQ', '96TFJT9N', '99S2T316', '9GQLZ50J', '9J08HQPH', '9JVGELXV', '9KZGDZ31', '9S3B2GYS', '9XQH3PDE', 'B36347PX', 'B3GWHYLV', 'B4H2WWYS', 'B53617TZ', 'B55FVXHR', 'B773WB96', 'B7K4FC8F', 'BDV32518', 'BEQH0M6Z', 'BGTW9DDB', 'BGYXB8PE', 'BS6Z00T7', 'BTEQP2RC', 'BTYB2XRN', 'BXH97S28', 'C4303LEZ', 'CC6T7Q3D', 'CD624N37', 'CKWMJ63M', 'CLXBXFXL', 'CMKXWTZQ', 'CNY1DG3E', 'CPV0FW0X', 'CSJM9SYS', 'CSY5ZS5M', 'CV81GEND', 'CZ868S7G', 'D23P3T99', 'D438Y3DS', 'D8NGFVRN', 'D988Q8E2', 'D9J5NQD4', 'DB60ZF3T', 'DESJCKHZ', 'DESY9QVK', 'DKVQZJQB', 'DMNTGHCF', 'DNV974CH', 'DQ1KZS23', 'DR65XZXF', 'DV87FE07', 'DX396B06', 'DZ3LV2YD', 'E1NFC1S6', 'E59VEB3P', 'E79M532Q', 'E98DBVEJ', 'EG39S37N', 'EM31SVY0', 'EQ6N37P7', 'ER85PYPC', 'ERZKNJ8N', 'ET2E5PNV', 'EXD4ZWVJ', 'EXGX32D1', 'EYD9P9CG', 'EYL75Y16', 'EYV5C4W9', 'F3F1BWX8', 'F4D2EXQV', 'F66QYR1V', 'F98MKK24', 'FCZHPS7W', 'FECLC9C3', 'FNWW0FJY', 'FV7187CN', 'FXS93NQW', 'G4NGBVGC', 'G99H39ZY', 'GCGPRX9R', 'GER2JMQK', 'GESKBZTW', 'GM1XM9FD', 'GN1Y43C0', 'GN3S2MG1', 'GPHH8T0K', 'GSQ4YHPT', 'GSQG36ZD', 'GTVSY43P', 'GW0SVPYB', 'GY9RY27K', 'GYK64FYG', 'GYR5QB5F', 'GZDRBV14', 'GZXJ6JRM', 'H30JWN6D', 'H9FLMRRZ', 'HCSKCG2C', 'HCWWHSWN', 'HCZ0G56T', 'HG5089NV', 'HJWFVZ5S', 'HLE56YEK', 'HPHPMSX0', 'HRJD61CS', 'HTZV1K7F', 'HWL3Z61P', 'HWNPCZTZ', 'HZK7VTSX', 'J0091SM9', 'J0CRCXC9', 'J1SVY7SL', 'J56H4P4J', 'JJ3FZ0LJ', 'JJ5H8T2H', 'JJPL15PT', 'JK6M376V', 'JTCRW1RE', 'JYCZ7JKG', 'K0K723RQ', 'K1C0MXT6', 'K20JBZL4', 'K3VNKN2P', 'K5HNZJ43', 'K5XG2RM5', 'KC255YFS', 'KH21JVZD', 'KJDEKVJH', 'KJTGBZN9', 'KK0CVCCJ', 'KLYHLG03', 'KNDBGZC2', 'KP8TFZE3', 'KS0WTZJ2', 'KSJXB523', 'L0V77BWT', 'L3S1DSH6', 'L48TG3EW', 'L4PXNEW9', 'L5VB63QR', 'L7NTJ5TP', 'LDZ41FK1', 'LJFTZB0F', 'LQDX07HX', 'LRQVTY7B', 'LS4M6FMC', 'LTD7WRRS', 'LWDDDT5V', 'M1GQXFYB', 'M2G1R7CQ', 'M4ZYPTP2', 'M53EJ7KW', 'M9XZMW52', 'MB6R5PRR', 'MBCKCXPB', 'MBS5HSL2', 'MEHNHETQ', 'MHJ1435L', 'MK4YLEW9', 'MKTGC2QZ', 'MLQFJYN1', 'MLZEPGWG', 'MMMLTS10', 'MP1CXKGN', 'MS7PBQ8Z', 'MTXS2X57', 'MXMH1384', 'N0H2F4RM', 'N19M957N', 'N64CKZ1Y', 'ND2Y0LRD', 'NEFMHNET', 'NFC557NB', 'NG2Y4762', 'NLG0YH5M', 'NQ3PWGMG', 'NV1G9BPH', 'NXNESHML', 'NZGZJ2P1', 'P1JF1QND', 'P3JEM049', 'P80QX3EB', 'P9LPDJJ2', 'PC0DTBM6', 'PCBJGQ8V', 'PCKXDFEC', 'PH2YLV53', 'PHDFWCCN', 'PME0SYMW', 'PNCMZGJB', 'PPCXJMTX', 'PQ9HQNVN', 'PQB3YEY5', 'PTCMMM63', 'PTWHCVNN', 'PYDRNXT1', 'Q11GFQMK', 'Q3TSJ19Q', 'Q5HRM4MF', 'Q9KQ4DD7', 'QENNM5V1', 'QJ8CMNDE', 'QM592V11', 'QNFDEWEB', 'QRBL4CDB', 'QS4JHQBC', 'QVN33L82', 'QWZMHPTJ', 'QZY771BV', 'R03CVQFW', 'R0BT2KNH', 'R0W20KLM', 'R3W54FJV', 'R3XPVFMF', 'R4VLPN8E', 'R63KWDBK', 'R6BP8FQ1', 'R880T4GS', 'R99B343J', 'RBJZ2XC4', 'REH8PHQ2', 'REQPDYJ6', 'RF2CF3B9', 'RHMPM70H', 'RJYGYJS9', 'RKR5TLWL', 'RN7HW77Q', 'RNY39ZWS', 'RTPZ3ZM8', 'RX61Q59G', 'S1NGF46X', 'S321VEP3', 'S5NG9K82', 'S5V09VQR', 'S5ZWZ7HW', 'S6SSMCK4', 'S85E8HZT', 'S9T0S78T', 'SFC8E2KF', 'SFKC682M', 'SH1YT7BM', 'SJBF10CK', 'SK9BGJ8E', 'SNHEKHZZ', 'SXRG95M6', 'SXY9HY6N', 'T1C668E6', 'T2J8G904', 'T3E9D98L', 'T84CH62W', 'TNTPLKMS', 'TQNDQQY7', 'TWL2BW0Z', 'V1ST6K5P', 'V8DRGT1B', 'V9DTWFWE', 'VHXPQ5P2', 'VL3L6GFC', 'VM0F6LD8', 'VNMV50V8', 'VV5D5RYK', 'W04N5CXL', 'W0X24CL8', 'W2DH820N', 'W7YZ8CSE', 'W8PL9MW5', 'WG0HPG7G', 'WHDM784R', 'WKR5CWBB', 'WQMGYPEG', 'WSFHSJ1C', 'WWH6LCNB', 'WYB4NG8H', 'X5Z8G6EH', 'X6PF2Z3C', 'X7FWTST5', 'XBV2CVLN', 'XE6HLH8Q', 'XL95LERW', 'XQ95WF3M', 'XRXHTW8G', 'XSV8CB8R', 'XT8CBPT4', 'XW6DKMEE', 'XX98BZF6', 'Y0423ZKN', 'Y0LSZL6S', 'Y1WFSTMH', 'Y416J3WM', 'Y5W4SKGM', 'Y7KK68KD', 'Y9GW7HPB', 'YBMLGSFB', 'YCV4YMKR', 'YE49YKQX', 'YLMRL2LD', 'YRCV19C9', 'YTP52KP4', 'YVGNY6R8', 'YW9VT16T', 'YWHC8ZWG', 'YY9XFE4C', 'YZKGS5C3', 'YZKZ0H0H', 'ZDBGQ34E', 'ZF8XNQ6L', 'ZHWMEY2V', 'ZLFB77TL', 'ZLXV6L4G', 'ZN0WHNPD', 'ZQ0LRN2M'
)

In [0]:
WITH t1 AS (
  SELECT DISTINCT 
      patient_id, 
      (YEAR(CURRENT_DATE) - YEAR(patient_yob)) AS age
  FROM (
    SELECT DISTINCT patient_id, patient_yob
    FROM (
      SELECT
        PATIENT_ID AS patient_id,
        PATIENT_YOB AS patient_yob,
        ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY PATIENT_YOB DESC) AS rn
      FROM com_raw.kom_patient_demographics
    )
    WHERE rn = 1
  )
),

bucketed AS (
  SELECT 
      patient_id,
      CASE 
          WHEN age BETWEEN 0 AND 5 THEN '0-5'
          WHEN age BETWEEN 6 AND 11 THEN '6-11'
          WHEN age BETWEEN 12 AND 16 THEN '12-16'
          WHEN age = 17 THEN '17'
          WHEN age BETWEEN 18 AND 20 THEN '18-20'
          WHEN age BETWEEN 21 AND 30 THEN '21-30'
          WHEN age BETWEEN 31 AND 50 THEN '31-50'
          WHEN age > 50 THEN '50+'
      END AS age_bucket
  FROM t1
  WHERE patient_id IN ('V8DRGT1B','07F040KL','CLXBXFXL','Y0423ZKN','FNWW0FJY','RN7HW77Q','4Q9YWTW2','GCGPRX9R','1181555','CPV0FW0X','DKVQZJQB','VL3L6GFC','LJFTZB0F','35RJEZJM','GZXJ6JRM','M9XZMW52','HLE56YEK','8WW54LY8','Q3TSJ19Q','RF2CF3B9','NFC557NB','6XZ63TZJ','BDV32518','C4303LEZ','7BDN8F8G','4RC079JT','YE49YKQX','DB60ZF3T','PCBJGQ8V','9GQLZ50J','GN3S2MG1','NZGZJ2P1','KJTGBZN9','GW0SVPYB','55MXFFR4','DNV974CH','QZY771BV','E98DBVEJ','ET2E5PNV','8TQK891K','E79M532Q','WWH6LCNB','5WQ44ZEV','B36347PX','CNY1DG3E','XW6DKMEE','HWL3Z61P','K5HNZJ43','KLYHLG03','2TQZ1H93','5HREMYP4','QNFDEWEB','RNY39ZWS','S1NGF46X','42D8R98N','4J55LN3N','G4NGBVGC','SXRG95M6','T3E9D98L','9JVGELXV','NQ3PWGMG','8LSZ647B','9S3B2GYS','CMKXWTZQ','MP1CXKGN','SH1YT7BM','SJBF10CK','1G84ST3K','4R9JZEPZ','4XQDPX1J','GYR5QB5F','LQDX07HX','87CS8ZN7','D9J5NQD4','LDZ41FK1','M1GQXFYB','R4VLPN8E','T2J8G904','0CFBS3QP','5GXK95K8','6KB07SBV','78VMNREV','8W4Z8SHZ','ER85PYPC','JTCRW1RE','JYCZ7JKG','R3W54FJV','1X7KPMBS','YLMRL2LD','JJPL15PT','MEHNHETQ','W0X24CL8','XBV2CVLN','4K210EEE','7J3SLYFW','BGYXB8PE','GY9RY27K','M2G1R7CQ','S9T0S78T','XE6HLH8Q','YWHC8ZWG','6MJEMWFT','84HGEGMM','B55FVXHR','BEQH0M6Z','PPCXJMTX','YBMLGSFB','ZDBGQ34E','1ETSLF32','3NSV0MXJ','5YQL634J','7FZHNZ49','85NWTEG1','B53617TZ','CD624N37','D438Y3DS','KNDBGZC2','MLQFJYN1','MXMH1384','RTPZ3ZM8','XT8CBPT4','4XK3PWRM','CSY5ZS5M','CV81GEND','DESJCKHZ','HRJD61CS','K5XG2RM5','L3S1DSH6','RJYGYJS9','SFKC682M','YW9VT16T','1WTYQP0W','2J7TTMMJ','47DCKLNL','6LRJE8F5','B3GWHYLV','EM31SVY0','J56H4P4J','MLZEPGWG','RKR5TLWL','SXY9HY6N','Y7KK68KD','2193XXGH','5S98ZESL','6513VL4S','96TFJT9N','DQ1KZS23','EQ6N37P7','EYL75Y16','GYK64FYG','H9FLMRRZ','HZK7VTSX','K3VNKN2P','PTWHCVNN','Q11GFQMK','W2DH820N','Y0LSZL6S','Y9GW7HPB','4NNDX85F','72R9ZN63','DZ3LV2YD','EG39S37N','F98MKK24','GTVSY43P','K0K723RQ','K20JBZL4','PNCMZGJB','QS4JHQBC','REH8PHQ2','REQPDYJ6','0RQ8WER6','2XPN1XBS','58D46PD2','7SBZKMP0','B4H2WWYS','CZ868S7G','D23P3T99','E1NFC1S6','F66QYR1V','FXS93NQW','G99H39ZY','M53EJ7KW','PH2YLV53','PTCMMM63','R63KWDBK','X6PF2Z3C','X7FWTST5','XRXHTW8G','XSV8CB8R','06NQHE5W','077VYZSG','2W1LYFK2','33PEP1TE','3EN349ET','5B5YJ0BJ','5B9F3PEP','5SR3QR8X','7K2MLVSZ','7MKVQFF5','E59VEB3P','GN1Y43C0','J0CRCXC9','LS4M6FMC','NG2Y4762','NLG0YH5M','RHMPM70H','S6SSMCK4','YZKZ0H0H','1PGKGH5D','47NW3JF6','5B1S5Y6M','7M0D8HCK','CSJM9SYS','EXD4ZWVJ','MTXS2X57','P80QX3EB','Q9KQ4DD7','QM592V11','QRBL4CDB','SK9BGJ8E','YY9XFE4C','ZN0WHNPD','165FCRS8','3L66Z8WX','755R9342','CKWMJ63M','D988Q8E2','DMNTGHCF','GM1XM9FD','L7NTJ5TP','MK4YLEW9','MS7PBQ8Z','N0H2F4RM','S321VEP3','WHDM784R','YRCV19C9','ZQ0LRN2M','020CLMY5','03DRQPXS','0RT0BC16','28KXJ10D','2H51L6TJ','7F3HFLSR','8ZHEJWP3','99S2T316','DV87FE07','JK6M376V','MKTGC2QZ','NEFMHNET','NXNESHML','S5V09VQR','S85E8HZT','3KFLJ9H5','5D4NL3WG','7TH16LJF','8KDEH0H8','92M2CVGQ','B773WB96','GPHH8T0K','HPHPMSX0','M4ZYPTP2','MHJ1435L','V9DTWFWE','VNMV50V8','X5Z8G6EH','XL95LERW','2L9ZH96X','508GEBRX','52ZYHDNZ','5KPZNFGX','80YQPQ2T','BGTW9DDB','BXH97S28','KC255YFS','KP8TFZE3','R0BT2KNH','R0W20KLM','S5NG9K82','SFC8E2KF','TWL2BW0Z','W04N5CXL','YCV4YMKR','YZKGS5C3','56Q7B70X','6B0EWF5Q','7W3FYDPQ','8B1MFBXC','8QR2GY9W','B7K4FC8F','EYD9P9CG','HCZ0G56T','K1C0MXT6','LRQVTY7B','N19M957N','N64CKZ1Y','TNTPLKMS','TQNDQQY7','W8PL9MW5','WSFHSJ1C','06WGTFBV','357ZX8QE','6MGD17QW','6Q831L53','73YZJ6P0','8NNK4CH8','DESY9QVK','GSQ4YHPT','GSQG36ZD','JJ3FZ0LJ','L5VB63QR','R03CVQFW','S5ZWZ7HW','0JXVTG3V','0VQR3EKL','36LNY5TR','8ML7WJG8','BTYB2XRN','HJWFVZ5S','J0091SM9','PYDRNXT1','RBJZ2XC4','XQ95WF3M','ZHWMEY2V','ZLFB77TL','39BDW50Z','F3F1BWX8','KK0CVCCJ','L48TG3EW','MMMLTS10','EXGX32D1','F4D2EXQV','GZDRBV14','KS0WTZJ2','P9LPDJJ2','PQ9HQNVN','VHXPQ5P2','Y416J3WM','596XQZSB','GESKBZTW','HCWWHSWN','HTZV1K7F','J1SVY7SL','QENNM5V1','QVN33L82','RX61Q59G','W7YZ8CSE','1TKZ01BN','EYV5C4W9','GER2JMQK','H30JWN6D','L0V77BWT','YTP52KP4','0FD2MVJH','HCSKCG2C','PCKXDFEC','Q5HRM4MF','WKR5CWBB','00P0S2C0','KH21JVZD','YVGNY6R8'
)
)

SELECT 
    age_bucket,
    COUNT(DISTINCT patient_id) AS patient_count
FROM bucketed
GROUP BY age_bucket
ORDER BY 
    CASE 
        WHEN age_bucket = '0-5' THEN 1
        WHEN age_bucket = '6-11' THEN 2
        WHEN age_bucket = '12-16' THEN 3
        WHEN age_bucket = '17' THEN 4
        WHEN age_bucket = '18-20' THEN 5
        WHEN age_bucket = '21-30' THEN 6
        WHEN age_bucket = '31-50' THEN 7
        WHEN age_bucket = '50+' THEN 8
    END;